In [1]:
import os

os.environ["LANGSMITH_TRACING"] = "true"  # 若 .env 里已有可省略
os.environ["LANGSMITH_PROJECT"] = "research_agent_final_test"  # 新 project 名，随意起

In [2]:
from src.graphs.main_graph import research_subgraph
from src.reporting.research_report import (
    build_batch_summary,
    export_batch_summary_html,
    export_research_eval_jsons,
    format_batch_summary,
)

In [3]:
import json
from pathlib import Path

TEST_DATA_DIR = Path("data/test data")

with (TEST_DATA_DIR / "sufficient_companies.json").open(encoding="utf-8") as handle:
    sufficient_data = json.load(handle)
with (TEST_DATA_DIR / "insufficient_companies.json").open(encoding="utf-8") as handle:
    insufficient_data = json.load(handle)

satisfied_companies = [
    {"name": company["name"], "gold_sufficient": True, "gold_satisfied": True}
    for company in sufficient_data["satisfied"][:20]
]
unsatisfied_companies = [
    {"name": company["name"], "gold_sufficient": True, "gold_satisfied": False}
    for company in sufficient_data["unsatisfied"]
]
insufficient_companies = [
    {"name": company["name"], "gold_sufficient": False, "gold_satisfied": None}
    for company in insufficient_data["insufficient_companies"][:20]
]

test_companies = [
    *satisfied_companies,
    *unsatisfied_companies,
    *insufficient_companies,
]
companies = test_companies
gold_by_company = {
    company["name"]: {
        "sufficient": company["gold_sufficient"],
        "satisfied": company["gold_satisfied"],
    }
    for company in test_companies
}
print(
    f"test_companies: {len(test_companies)} "
    f"(satisfied {len(satisfied_companies)}, "
    f"unsatisfied {len(unsatisfied_companies)}, "
    f"insufficient {len(insufficient_companies)})"
)

requirement = "A meeting room or event space on the property that provides seating for group events."
# requirement = "The property must have a gym"
positive_when = "There is at least one on-site meeting or event room in a single room that can seat 20 or more people."
negative_when = "There is no on-site meeting or event room that can seat 20 or more people in a single room."
does_not_count = [
    "nearby or off-site venues or partners",
    "combined capacity across multiple rooms",
    "a sibling or different property",
    "guest house rules (party/pets/smoking/quiet hours/check-in)",
    "not found online (absence of evidence is not a fail)",
]
skills = ["meetings"]

inputs = [
    {
        "company": company["name"],
        "requirement": requirement,
        "positive_when": positive_when,
        "negative_when": negative_when,
        "does_not_count": does_not_count,
        "skills": skills,
    }
    for company in companies
]

results = await research_subgraph.abatch(
    inputs,
    config={"max_concurrency": 10},
    return_exceptions=True,
)

test_companies: 56 (satisfied 20, unsatisfied 16, insufficient 20)


In [4]:
from llm.models import RESEARCH_LLM_MODEL

summary = build_batch_summary(inputs, results, gold_by_company=gold_by_company)
model = RESEARCH_LLM_MODEL
batch_size = len(companies)

report_path = export_batch_summary_html(
    summary,
    f"reports/test_final/{batch_size}_companies/model_{model}_research_report.html",
    title="Hotel Research Report",
)
print(f"HTML report saved to: {report_path.resolve()}")

HTML report saved to: D:\B2B_lead_gen\reports\test_final\56_companies\model_gpt-5-mini_research_report.html


In [5]:
eval_dir = f"data/research_evals/final_{len(companies)}_companies"
written = export_research_eval_jsons(inputs, results, eval_dir)
for name, path in written.items():
    print(f"{name}: {path.resolve()}")

query_generator_decisions: D:\B2B_lead_gen\data\research_evals\final_56_companies\query_generator_decisions.json
url_selector_decisions: D:\B2B_lead_gen\data\research_evals\final_56_companies\url_selector_decisions.json
evidence_extractor_decisions: D:\B2B_lead_gen\data\research_evals\final_56_companies\evidence_extractor_decisions.json
verifier_judgments: D:\B2B_lead_gen\data\research_evals\final_56_companies\verifier_judgments.json
unclear_ownership_judgments: D:\B2B_lead_gen\data\research_evals\final_56_companies\unclear_ownership_judgments.json
research_agent_steps: D:\B2B_lead_gen\data\research_evals\final_56_companies\research_agent_steps.json
research_outcomes: D:\B2B_lead_gen\data\research_evals\final_56_companies\research_outcomes.json
